# Lab 2.11 &mdash; Image Classification on MNIST

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Introduction to Deep Learning**

### What you'll do
- Train a network to classify handwritten digits (MNIST)
- Hold out a validation split and capture the training history
- Plot train vs validation curves and reach high accuracy

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 2 slides &mdash; The same model, two frameworks](../../presentation/day1-module2-introduction-to-deep-learning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 2 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # quiet TensorFlow logs (used in the advanced labs)
WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-02-11")
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
The classic deep-learning 'hello world': **MNIST** handwritten digits. You'll train a network,
keep a **validation split** to watch generalisation, and plot the **training curves**.

> Needs `tensorflow` and `matplotlib`. Target: MNIST; offline fallback to 8x8 digits if needed.

In [ ]:
# Data loader: real MNIST in the sandbox, sklearn digits as an offline fallback.
import numpy as np
def load_image_data(n_train=12000, n_test=2000):
    try:
        from tensorflow.keras.datasets import mnist
        (xtr, ytr), (xte, yte) = mnist.load_data()
        xtr = xtr.reshape(len(xtr), -1).astype("float32") / 255.0
        xte = xte.reshape(len(xte), -1).astype("float32") / 255.0
        name, side = "MNIST (28x28)", 28
    except Exception:
        from sklearn.datasets import load_digits
        d = load_digits(); X = (d.data / 16.0).astype("float32"); y = d.target
        xtr, ytr, xte, yte = X[:1400], y[:1400], X[1400:], y[1400:]
        name, side = "sklearn digits (8x8, offline fallback)", 8
    return (xtr[:n_train], ytr[:n_train]), (xte[:n_test], yte[:n_test]), name, side, xtr.shape[1]

(X_tr, y_tr), (X_te, y_te), DATA_NAME, SIDE, NFEAT = load_image_data()
print("dataset:", DATA_NAME, "| train:", X_tr.shape, "| test:", X_te.shape)

## Your Turn
Set the **validation split** and **epochs**, fit while capturing `history`, then read the final
validation accuracy.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build():
    m = keras.Sequential([layers.Input((NFEAT,)),
                          layers.Dense(128, activation='relu'),
                          layers.Dense(10, activation='softmax')])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

model = None; history = None; test_acc = None; final_val_acc = None
try:
    model = build()
    history = model.fit(X_tr, y_tr, validation_split=___, epochs=___,
                        batch_size=128, verbose=0)
    test_acc = float(model.evaluate(X_te, y_te, verbose=0)[1])
    final_val_acc = ___   # TODO: last value of history.history["val_accuracy"]
    print('test accuracy:', round(test_acc, 3), '| final val accuracy:', round(final_val_acc, 3))
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("training captured a validation curve", lambda: "val_accuracy" in history.history)
expect_true("test accuracy >= 0.90", lambda: test_acc >= 0.90)
expect_true("final validation accuracy read correctly", lambda: abs(final_val_acc - history.history["val_accuracy"][-1]) < 1e-9)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Bonus: plot the training curves (not graded)
Run this to see accuracy and loss for train vs validation across epochs.

In [ ]:
try:
    import matplotlib.pyplot as plt
    h = history.history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
    ax1.plot(h["accuracy"], label="train"); ax1.plot(h["val_accuracy"], label="val")
    ax1.set_title("Accuracy"); ax1.set_xlabel("epoch"); ax1.legend()
    ax2.plot(h["loss"], label="train"); ax2.plot(h["val_loss"], label="val")
    ax2.set_title("Loss"); ax2.set_xlabel("epoch"); ax2.legend()
    plt.tight_layout(); plt.savefig(WORK + "/mnist_curves.png", dpi=90); plt.show()
    print("saved:", WORK + "/mnist_curves.png")
except Exception as e:
    print("Plot needs matplotlib + a trained model.", type(e).__name__)

---
### Key takeaway
You trained a real digit classifier and watched it generalise. Next, the capstone: open the box and *see how it decides*.

[&#8592; All Module 2 labs](./index.html) &nbsp;&middot;&nbsp; [Module 2 slides](../../presentation/day1-module2-introduction-to-deep-learning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>